# Entraînement RNN sur les 6 variantes SIEVE

Ce notebook entraîne un RNN indépendamment sur chaque variante (`output_0` à `output_0.5`), puis sauvegarde les modèles et un tableau comparatif.

**Prérequis :** les fichiers préparés par le notebook de prétraitement doivent être dans le même dossier :
`output_0_train.csv`, `output_0_val.csv`, `output_0_test.csv`, etc.

Chaque fichier doit contenir les colonnes `log` et `label`.


In [9]:
import os
import re
import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

print("Appareil utilisé :", DEVICE)

VARIANTES = ["output_0", "output_0.1", "output_0.2",
             "output_0.3", "output_0.4", "output_0.5"]

BATCH_SIZE = 32
EPOCHS = 3
MAX_LENGTH = 128
EMBEDDING_DIM = 128
HIDDEN_DIM = 128
LEARNING_RATE = 1e-3

os.makedirs("resultats_rnn", exist_ok=True)


Appareil utilisé : mps


In [5]:
def tokenizer(texte):
    # On conserve les mots, nombres, adresses IP et signes utiles des logs.
    return re.findall(r"[a-z]+|\\d+(?:\\.\\d+){1,3}|\\d+|[^\\w\\s]", str(texte).lower())


class LogsDataset(Dataset):
    def __init__(self, textes, labels, vocab):
        self.textes = textes
        self.labels = labels
        self.vocab = vocab

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        tokens = tokenizer(self.textes.iloc[idx])
        ids = [self.vocab.get(tok, self.vocab["<UNK>"]) for tok in tokens]
        ids = ids[:MAX_LENGTH]
        longueur = max(1, len(ids))
        ids = ids + [self.vocab["<PAD>"]] * (MAX_LENGTH - len(ids))
        return (
            torch.tensor(ids, dtype=torch.long),
            torch.tensor(longueur, dtype=torch.long),
            torch.tensor(int(self.labels.iloc[idx]), dtype=torch.long)
        )


class RNNClassifier(nn.Module):
    def __init__(self, vocab_size, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, EMBEDDING_DIM, padding_idx=0)
        self.rnn = nn.RNN(EMBEDDING_DIM, HIDDEN_DIM, batch_first=True)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(HIDDEN_DIM, num_classes)

    def forward(self, x, lengths):
        embedded = self.embedding(x)
        output, _ = self.rnn(embedded)
        # Récupérer la sortie du dernier vrai token, pas du padding.
        indices = (lengths - 1).to(x.device)
        last_output = output[torch.arange(x.size(0), device=x.device), indices]
        return self.fc(self.dropout(last_output))


def construire_vocab(train_textes):
    vocab = {"<PAD>": 0, "<UNK>": 1}
    for texte in train_textes:
        for token in tokenizer(texte):
            if token not in vocab:
                vocab[token] = len(vocab)
    return vocab


def creer_loader(df, vocab, shuffle=False):
    dataset = LogsDataset(df["log"], df["label"], vocab)
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=0)


def evaluer_modele(model, loader, criterion=None):
    model.eval()
    pertes, vraies, predictions = [], [], []

    with torch.no_grad():
        for x, longueurs, y in loader:
            x, longueurs, y = x.to(DEVICE), longueurs.to(DEVICE), y.to(DEVICE)
            sorties = model(x, longueurs)
            if criterion is not None:
                pertes.append(criterion(sorties, y).item())
            pred = torch.argmax(sorties, dim=1)
            vraies.extend(y.cpu().numpy())
            predictions.extend(pred.cpu().numpy())

    accuracy = accuracy_score(vraies, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        vraies, predictions, average="macro", zero_division=0
    )
    perte = float(np.mean(pertes)) if pertes else None
    return perte, accuracy, precision, recall, f1, vraies, predictions


## Entraînement automatique des six variantes

In [6]:
import csv
from pathlib import Path

DATA_DIR = next(
    (
        dossier
        for dossier in (Path("../processed_data"), Path("processed_data"))
        if (dossier / f"{VARIANTES[0]}_X_train.csv").is_file()
    ),
    None,
)
if DATA_DIR is None:
    raise FileNotFoundError("Impossible de trouver le dossier processed_data.")


def charger_split(path):
    lignes = []
    with path.open("r", encoding="utf-8", newline="") as fichier:
        lecteur = csv.reader(fichier)
        entete = next(lecteur, None)
        if entete != ["log", "label"]:
            raise ValueError(f"En-tête inattendu dans {path} : {entete}")

        for numero, ligne in enumerate(lecteur, start=2):
            if not ligne:
                continue
            if len(ligne) < 2:
                raise ValueError(f"Ligne {numero} invalide dans {path}")
            log = ",".join(ligne[:-1]).strip()
            try:
                label = int(ligne[-1].strip())
            except ValueError as exc:
                raise ValueError(
                    f"Label invalide à la ligne {numero} dans {path}"
                ) from exc
            lignes.append({"log": log, "label": label})

    return pd.DataFrame(lignes, columns=["log", "label"])


resultats = []

for variante in VARIANTES:
    print("\\n" + "=" * 60)
    print("VARIANTE :", variante)
    print("=" * 60)

    train_path = DATA_DIR / f"{variante}_X_train.csv"
    val_path = DATA_DIR / f"{variante}_X_val.csv"
    test_path = DATA_DIR / f"{variante}_X_test.csv"

    for chemin in [train_path, val_path, test_path]:
        if not os.path.exists(chemin):
            raise FileNotFoundError(
                f"Fichier manquant : {chemin}. Vérifie les fichiers produits par le prétraitement."
            )

    train_df = charger_split(train_path)
    val_df = charger_split(val_path)
    test_df = charger_split(test_path)

    # Vérification des colonnes attendues
    for nom, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
        if not {"log", "label"}.issubset(df.columns):
            raise ValueError(f"{variante}_{nom}.csv doit contenir les colonnes 'log' et 'label'.")

    # Construire le vocabulaire uniquement avec les données d'entraînement
    vocab = construire_vocab(train_df["log"])
    num_classes = int(train_df["label"].nunique())

    train_loader = creer_loader(train_df, vocab, shuffle=True)
    val_loader = creer_loader(val_df, vocab)
    test_loader = creer_loader(test_df, vocab)

    model = RNNClassifier(len(vocab), num_classes).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    meilleur_f1_val = -1
    meilleur_etat = None

    for epoch in range(EPOCHS):
        model.train()
        pertes_train = []

        for x, longueurs, y in train_loader:
            x, longueurs, y = x.to(DEVICE), longueurs.to(DEVICE), y.to(DEVICE)

            optimizer.zero_grad()
            sorties = model(x, longueurs)
            loss = criterion(sorties, y)
            loss.backward()
            optimizer.step()
            pertes_train.append(loss.item())

        val_loss, val_acc, val_precision, val_recall, val_f1, _, _ = evaluer_modele(
            model, val_loader, criterion
        )

        print(
            f"Epoch {epoch+1}/{EPOCHS} | "
            f"Train loss: {np.mean(pertes_train):.4f} | "
            f"Val loss: {val_loss:.4f} | "
            f"Val accuracy: {val_acc:.4f} | Val macro-F1: {val_f1:.4f}"
        )

        if val_f1 > meilleur_f1_val:
            meilleur_f1_val = val_f1
            meilleur_etat = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    # Restaurer le meilleur modèle selon le F1 de validation
    if meilleur_etat is not None:
        model.load_state_dict(meilleur_etat)

    test_loss, test_acc, test_precision, test_recall, test_f1, y_true, y_pred = evaluer_modele(
        model, test_loader, criterion
    )

    print(f"TEST | Accuracy: {test_acc:.4f} | Macro-F1: {test_f1:.4f}")
    print(classification_report(y_true, y_pred, zero_division=0))

    # Sauvegarder le modèle et le vocabulaire de cette variante
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "vocab_size": len(vocab),
            "num_classes": num_classes,
            "max_length": MAX_LENGTH,
            "embedding_dim": EMBEDDING_DIM,
            "hidden_dim": HIDDEN_DIM
        },
        f"resultats_rnn/rnn_{variante}.pth"
    )

    with open(f"resultats_rnn/vocab_{variante}.json", "w", encoding="utf-8") as f:
        json.dump(vocab, f, ensure_ascii=False)

    resultats.append({
        "variante": variante,
        "accuracy": test_acc,
        "precision_macro": test_precision,
        "recall_macro": test_recall,
        "f1_macro": test_f1,
        "test_loss": test_loss
    })

    # Sauvegarde après chaque variante : les résultats déjà obtenus sont conservés
    pd.DataFrame(resultats).to_csv("resultats_rnn/comparaison_rnn.csv", index=False)

print("\\nEntraînement des six variantes terminé.")


\n============================================================
VARIANTE : output_0
Epoch 1/3 | Train loss: 1.3274 | Val loss: 0.5870 | Val accuracy: 0.8023 | Val macro-F1: 0.7956
Epoch 2/3 | Train loss: 0.4447 | Val loss: 0.2870 | Val accuracy: 0.9013 | Val macro-F1: 0.9013
Epoch 3/3 | Train loss: 0.2967 | Val loss: 0.3709 | Val accuracy: 0.8760 | Val macro-F1: 0.8751
TEST | Accuracy: 0.8973 | Macro-F1: 0.8971
              precision    recall  f1-score   support

           0       0.53      0.51      0.52       175
           1       0.60      0.62      0.61       175
           2       0.67      0.77      0.71       175
           3       0.91      0.85      0.88       175
           4       0.69      0.85      0.76       175
           5       0.99      0.89      0.94       175
           6       1.00      0.77      0.87       175
           7       1.00      1.00      1.00       175
           8       0.83      0.94      0.88       175
           9       0.99      1.00      1.00  

## Tableau comparatif final

In [8]:
comparaison = pd.DataFrame(resultats)
display(comparaison)
comparaison.to_csv("resultats_rnn/comparaison_rnn.csv", index=False)


,variante,accuracy,precision_macro,recall_macro,f1_macro,test_loss
0,output_0,0.897333,0.903022,0.897333,0.897096,0.287407
1,output_0.1,0.802286,0.815189,0.802286,0.799950,0.616201
2,output_0.2,0.709714,0.729276,0.709714,0.708770,0.930369
3,output_0.3,0.683810,0.694900,0.683810,0.678293,1.010964
4,output_0.4,0.617905,0.628696,0.617905,0.606702,1.228728
5,output_0.5,0.650476,0.663723,0.650476,0.643580,1.108755
